# 🧠 Redes Neuronales para Predecir Porosidad
## Módulo 6 · Día 1 · Construcción de un MLP con Keras · Capacitación SLB

**Instructor: David Ponce**

---

### 🎯 ¿Qué aprenderemos hoy?

Hasta ahora usamos modelos de ML clásico (K-Means, DBSCAN, PCA). Hoy damos
el salto a las **redes neuronales**: máquinas que aprenden patrones complejos
por sí mismas, sin que les demos las reglas.

Nuestra misión: **predecir la porosidad (NPHI) de la roca** a partir de otras
3 curvas geofísicas (GR, ILD, RHOB). Es un problema de **regresión** — predecir
un valor continuo.

### 📋 Objetivos de la sesión:
1. Entender qué es una neurona artificial (pesos + sesgo + activación)
2. Construir un Perceptrón Multicapa (MLP) con Keras
3. Entrenar la red para predecir porosidad y evaluar su error (MSE)
4. Visualizar la porosidad predicha vs la real

> 💡 **Tip:** Cada celda de código está comentada línea por línea. Lee los
> comentarios mientras ejecutas.

---
## 🧩 PARTE 1: Importando las herramientas

Hoy estrenamos **TensorFlow/Keras**, la librería de deep learning de Google.
Keras es una interfaz de alto nivel que hace que construir redes neuronales
sea tan simple como apilar bloques de Lego.

### 📦 Celda 1: Importación de librerías

In [ ]:
# ============================================
# CELDA 1: Importación de librerías
# ============================================

# ─── Herramientas clásicas (ya conocidas) ───
import numpy as np                          # Cálculo numérico
import pandas as pd                         # Tablas (DataFrames)
import matplotlib.pyplot as plt             # Gráficos
import seaborn as sns                       # Gráficos estadísticos

# ─── Preprocesamiento (ya conocidas) ───
from sklearn.preprocessing import StandardScaler
#   ↑ El escalador Z-Score. ¡OBLIGATORIO en redes neuronales!
from sklearn.model_selection import train_test_split
#   ↑ NUEVA: divide los datos en entrenamiento y validación.

# ─── TensorFlow / Keras (¡NUEVO HOY!) ───
import tensorflow as tf
#   ↑ 'tensorflow' = la plataforma de deep learning de Google.
#   ↑ 'as tf' = alias estándar.
from tensorflow.keras.models import Sequential
#   ↑ 'Sequential' = forma de construir redes apilando capas una tras otra.
#   ↑ Es el tipo de modelo más simple: una secuencia lineal de capas.
from tensorflow.keras.layers import Dense
#   ↑ 'Dense' = capa donde CADA neurona se conecta a TODAS las neuronas
#   ↑   de la capa anterior (también llamada 'fully connected').
#   ↑   Es la capa fundamental de un MLP.

# ─── Estética ───
sns.set_theme(style="whitegrid")
print("✅ Librerías importadas. TensorFlow versión:", tf.__version__)


### 🔍 ¿Qué hace cada librería NUEVA?

| Librería | ¿Qué es? | ¿Para qué la usamos HOY? |
|----------|----------|---------------------------|
| `tensorflow` | Plataforma de deep learning | El motor que entrena la red neuronal |
| `Sequential` | Modelo apilado | Construir la red capa por capa, en orden |
| `Dense` | Capa totalmente conectada | Cada neurona se conecta a todas las de la capa anterior |
| `train_test_split` | Divisor de datos | 80% para entrenar, 20% para validar |

> 🧠 **¿Por qué TensorFlow y no sklearn?** sklearn tiene ML clásico
> (K-Means, árboles, regresión). Para redes neuronales profundas, TensorFlow/Keras
> es el estándar de la industria.

---
## 📥 PARTE 2: Cargando y entendiendo los datos

Dataset: `registro_petrofisico.csv` — 7,000 registros de un pozo.
Cada registro tiene 4 curvas geofísicas + la profundidad.

Nuestra tarea: usar **3 variables de entrada** para predecir **1 variable objetivo**:
- Entradas (X): `GR_API`, `ILD_ohm_m`, `RHOB_g_cc`
- Objetivo (y): `NPHI_v_v` (porosidad neutrón)

### 📦 Celda 2: Cargar y explorar el dataset

In [ ]:
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_6_deep_learning/data/registro_petrofisico.csv -O registro_petrofisico.csv
# ============================================
# CELDA 2: Cargar el dataset petrofísico
# ============================================

df_well = pd.read_csv('registro_petrofisico.csv')
#   ↑ Lee el CSV y lo convierte en un DataFrame.

# ─── Exploración inicial ───
print("📊 Dimensiones:", df_well.shape)
#   ↑ Esperamos (7001, 5): 7,000 registros y 5 columnas.

print("\n📋 Tipos de datos:")
print(df_well.dtypes)
#   ↑ Todas deben ser float64 (números decimales).

print("\n📈 Estadísticas:")
df_well.describe()
#   ↑ Resumen estadístico de cada columna.
#   ↑ Mira los rangos: GR (10-120), ILD (0.5-60), RHOB (2.1-2.7), NPHI (0-0.35).

df_well.head()
#   ↑ Primeras 5 filas para verificar la estructura.


### 🔍 ¿Qué buscamos?

- `shape` → (7001, 5). Si es muy distinto, el CSV no cargó bien.
- `dtypes` → todas float64. Si alguna es object, hay texto infiltrado.
- `describe()` → los rangos de cada curva. Fíjate que **NPHI** (0 a 0.35)
  es mucho más pequeña que **GR** (10 a 120). Esto importará al escalar.


---
## 📏 PARTE 3: Escalar entradas Y salida

**Regla de oro:** las redes neuronales son MUY sensibles a la escala.
Si GR va de 10-120 y NPHI de 0-0.35, el optimizador tendrá problemas
para converger. Por eso escalamos TANTO las entradas como la salida.

> ⚠️ **Diferencia con los módulos anteriores:** antes solo escalábamos las
> entradas. En redes neuronales también escalamos la SALIDA (y), porque
> la red genera valores internos pequeños.

### 📦 Celda 3: Escalar y dividir los datos

In [ ]:
# ============================================
# CELDA 3: Escalar entradas y salida, dividir datos
# ============================================

# ─── Definir qué columnas son entrada y cuál es objetivo ───
features = ['GR_API', 'ILD_ohm_m', 'RHOB_g_cc']
#   ↑ 3 variables de ENTRADA (lo que la red "ve").
target = ['NPHI_v_v']
#   ↑ 1 variable OBJETIVO (lo que la red debe predecir).

# ─── Escalar entradas (X) ───
scaler_X = StandardScaler()
#   ↑ Crea el escalador para las entradas.
X_scaled = scaler_X.fit_transform(df_well[features])
#   ↑ Calcula μ y σ de cada entrada y aplica z = (x-μ)/σ.

# ─── Escalar salida (y) ───
scaler_y = StandardScaler()
#   ↑ Crea un escalador SEPARADO para la salida.
y_scaled = scaler_y.fit_transform(df_well[target])
#   ↑ Escala NPHI. Guardamos scaler_y porque lo usaremos para des-escalar
#   ↑   las predicciones al final (volver a las unidades reales).

# ─── Dividir en entrenamiento (80%) y validación (20%) ───
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled,
    test_size=0.2,       # 20% para validación
    random_state=42      # semilla fija para reproducibilidad
)
#   ↑ 'train_test_split' baraja los datos y los divide aleatoriamente.
#   ↑ Devuelve 4 objetos: X de entrenamiento, X de validación,
#   ↑   y de entrenamiento, y de validación.

print(f"📊 Entrenamiento: {X_train.shape[0]} registros")
print(f"📊 Validación: {X_val.shape[0]} registros")
#   ↑ 80% de 7000 = 5600 para entrenar, 1400 para validar.


### 🔍 ¿Por qué escalar la salida también?

La red neuronal usa gradientes para ajustar pesos. Si la salida real es 0.20
y la red predice 0.15, el error es pequeño. Si usáramos valores sin escalar
(NPHI en fracción 0.20 vs GR en 120), las escalas chocarían y el optimizador
no sabría cómo balancear. Escalar TODO a media=0, std=1 hace que la red
trabaje en un espacio uniforme.

---
## 🏗️ PARTE 4: Construyendo el MLP

Un MLP (Perceptrón Multicapa) es una pila de capas `Dense`.
Cada capa `Dense` conecta todas las neuronas de la capa anterior con las de la siguiente.

Arquitectura de hoy:
1. **Capa de entrada** (implícita): 3 neuronas (GR, ILD, RHOB)
2. **Capa oculta 1**: 64 neuronas, activación ReLU
3. **Capa oculta 2**: 32 neuronas, activación ReLU
4. **Capa de salida**: 1 neurona, activación lineal (para regresión)


### 📦 Celda 4: Construir el modelo

In [ ]:
# ============================================
# CELDA 4: Construir el MLP con Keras Sequential
# ============================================

model = Sequential([
    # ─── Capa oculta 1: 64 neuronas ───
    Dense(64, activation='relu', input_shape=(3,)),
    #   ↑ 'Dense(64)' = 64 neuronas en esta capa.
    #   ↑ 'activation=relu' = ReLU: si la suma es negativa, da 0;
    #   ↑   si es positiva, la pasa tal cual. Añade no-linealidad.
    #   ↑ 'input_shape=(3,)' = SOLO en la primera capa. Le dice a Keras
    #   ↑   que la entrada tiene 3 columnas (GR, ILD, RHOB).

    # ─── Capa oculta 2: 32 neuronas ───
    Dense(32, activation='relu'),
    #   ↑ Segunda capa oculta. No necesita input_shape: Keras lo deduce
    #   ↑   automáticamente de la capa anterior (64 neuronas).

    # ─── Capa de salida: 1 neurona ───
    Dense(1, activation='linear')
    #   ↑ 'Dense(1)' = 1 neurona de salida (predice UN solo valor: NPHI).
    #   ↑ 'activation=linear' = pasa la suma tal cual, sin transformar.
    #   ↑   Para REGRESIÓN (predecir un número continuo), usamos lineal.
    #   ↑   Para CLASIFICACIÓN usaríamos 'sigmoid' o 'softmax'.
])

model.summary()
#   ↑ Muestra la arquitectura: número de capas, neuronas y parámetros.
#   ↑ Observa cuántos parámetros entrenables tiene la red (los pesos + sesgos).


### 🔍 ¿Por qué ReLU en ocultas y Lineal en salida?

- **ReLU** (Rectified Linear Unit): si z > 0 → z, si z < 0 → 0.
  Añade no-linealidad, permitiendo a la red aprender patrones curvos.
  Es la activación más usada en capas ocultas.
- **Lineal**: devuelve z sin cambios. Para regresión queremos predecir
  cualquier valor continuo (porosidad), sin restringirlo a un rango [0,1].


### 📦 Celda 5: Compilar el modelo

In [ ]:
# ============================================
# CELDA 5: Compilar — definir optimizador y pérdida
# ============================================

model.compile(
    optimizer='adam',   # ← cómo la red ajusta sus pesos en cada paso
    loss='mse'          # ← error cuadrático medio: qué tan lejos está
                        #   la predicción del valor real
)
#   ↑ 'optimizer=adam' = el optimizador Adam (veremos más mañana).
#   ↑   Es el estándar: tasa de aprendizaje adaptativa.
#   ↑ 'loss=mse' = Mean Squared Error. Para regresión, es la pérdida estándar.
#   ↑   Penaliza errores grandes más que pequeños (al elevar al cuadrado).

print("✅ Modelo compilado.")


---
## 🚀 PARTE 6: Entrenando la red

Entrenar = repetir el ciclo **forward (predecir) → error → backpropagation (corregir)**
muchas veces. Cada pasada completa por todos los datos se llama **época**.


### 📦 Celda 6: Entrenar el modelo

In [ ]:
# ============================================
# CELDA 6: Entrenar la red por 50 épocas
# ============================================

history = model.fit(
    X_train, y_train,                       # ← datos de entrenamiento
    epochs=50,                              # ← 50 vueltas completas
    batch_size=32,                          # ← procesa 32 registros a la vez
    validation_data=(X_val, y_val),         # ← evalúa en validación cada época
    verbose=1                               # ← muestra el progreso
)
#   ↑ 'model.fit' = entrena la red.
#   ↑ 'epochs=50' = 50 pasadas completas por los datos de entrenamiento.
#   ↑ 'batch_size=32' = la red actualiza pesos cada 32 registros (mini-batch).
#   ↑ 'validation_data' = datos que la red NO usa para entrenar, solo para
#   ↑   medir qué tan bien generaliza. Esto es clave para detectar overfitting.
#   ↑ 'history' = objeto que guarda el historial de pérdida por época.


### 📦 Celda 7: Graficar la curva de pérdida

In [ ]:
# ============================================
# CELDA 7: Curva de pérdida (MSE) por época
# ============================================

plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Entrenamiento', linewidth=2)
#   ↑ 'history.history' = diccionario con las métricas registradas.
#   ↑ '['loss']' = la pérdida de entrenamiento por época.
plt.plot(history.history['val_loss'], label='Validación', linewidth=2)
#   ↑ '['val_loss']' = la pérdida de validación por época.
plt.title('Evolución de la Pérdida (MSE) durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('MSE (Error Cuadrático Medio)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
#   ↑ Si AMBAS curvas bajan juntas → la red aprende bien.
#   ↑ Si entrenamiento baja pero validación sube → OVERFITTING (Día 2).


### 🔍 ¿Qué esperamos ver?

Ambas curvas (entrenamiento y validación) deberían **bajar juntas** y
estabilizarse. Eso significa que la red aprendió la relación GR/ILD/RHOB → NPHI
y generaliza bien a datos nuevos.

> ⚠️ Si ves que la de validación empieza a SUBIR mientras la de entrenamiento
> sigue bajando, eso es **overfitting**. Lo atacaremos mañana.

---
## 🎯 PARTE 8: Predecir y des-escalar

La red predice en espacio escalado. Para comparar con la porosidad REAL,
debemos **des-escalar** (invertir la transformación Z-Score).

### 📦 Celda 8: Predecir y des-escalar

In [ ]:
# ============================================
# CELDA 8: Predecir porosidad y des-escalar
# ============================================

# ─── Predecir sobre los datos de validación ───
y_pred_scaled = model.predict(X_val)
#   ↑ 'model.predict' = la red genera predicciones para los datos de validación.
#   ↑ 'y_pred_scaled' = predicciones en espacio ESCALADO (media=0, std=1).

# ─── Des-escalar las predicciones ───
y_pred = scaler_y.inverse_transform(y_pred_scaled)
#   ↑ 'inverse_transform' = revierte la transformación Z-Score.
#   ↑   z → x: x = z * σ + μ. Volvemos a las unidades reales de NPHI (fracción).

# ─── Des-escalar los valores reales para comparar ───
y_val_real = scaler_y.inverse_transform(y_val)
#   ↑ También des-escalamos los valores reales de validación.

print(f"✅ Predicción completada: {y_pred.shape[0]} valores.")
print(f"   Rango de porosidad predicha: {y_pred.min():.3f} - {y_pred.max():.3f}")
print(f"   Rango de porosidad real: {y_val_real.min():.3f} - {y_val_real.max():.3f}")


### 🔍 ¿Por qué des-escalar?

La red aprende en espacio escalado (media=0, std=1) porque así converge mejor.
Pero un valor escalado de '0.5' no significa nada para un petrofísico. Al
des-escalar, volvemos a las unidades reales: porosidad como fracción (ej. 0.20
= 20% de porosidad).


### 📦 Celda 9: Comparar predicción vs realidad

In [ ]:
# ============================================
# CELDA 9: Visualizar porosidad predicha vs real
# ============================================

# ─── Recuperar la profundidad correspondiente a los datos de validación ───
_, X_val_idx, _, _ = train_test_split(
    df_well[['Profundidad_m']], y_scaled, test_size=0.2, random_state=42
)
#   ↑ Necesitamos la profundidad de los datos de validación para graficar.
#   ↑ Usamos el mismo random_state=42 para que la división coincida.

plt.figure(figsize=(10, 6))
plt.scatter(X_val_idx['Profundidad_m'], y_val_real, s=3, alpha=0.5, label='NPHI Real', color='#38bdf8')
#   ↑ Puntos azules: la porosidad REAL.
plt.scatter(X_val_idx['Profundidad_m'], y_pred, s=3, alpha=0.5, label='NPHI Predicha', color='#f59e0b')
#   ↑ Puntos naranjas: la porosidad PREDICHA por la red.
plt.xlabel('Profundidad (m)')
plt.ylabel('Porosidad NPHI (v/v)')
plt.title('Porosidad Real vs Predicha por la Red Neuronal')
plt.legend()
plt.show()

#   ↑ Si los puntos azules y naranjas se SOLAPAN → la red aprendió la física.
#   ↑ Si están dispersos y no coinciden → la red no aprendió bien.


---
## 📋 RECAP: El Pipeline MLP Completo

```
┌────────────────────────────────────────────────────────┐
│ 1. IMPORTAR      → tensorflow, Keras, sklearn          │
│ 2. CARGAR datos  → pd.read_csv()                       │
│ 3. ESCALAR X e y → StandardScaler() (¡ambos!)          │
│ 4. DIVIDIR       → train_test_split() 80/20            │
│ 5. CONSTRUIR     → Sequential([Dense, Dense, Dense])   │
│ 6. COMPILAR      → optimizer='adam', loss='mse'        │
│ 7. ENTRENAR      → model.fit(epochs=50)                │
│ 8. GRAFICAR      → curva de pérdida                    │
│ 9. PREDECIR      → predict() + inverse_transform()     │
└────────────────────────────────────────────────────────┘
```

### ✅ Lo que aprendiste hoy

- Una neurona = pesos + sesgo + activación
- Un MLP = capas Dense apiladas
- ReLU en capas ocultas, Lineal en salida de regresión
- Entrenar = repetir forward → error → backpropagation
- Escalar SIEMPRE (entradas Y salida)

> 🚀 **Siguiente:** Mañana veremos el enemigo #1 del deep learning: el
> overfitting. Y aprenderemos a combatirlo con Dropout y Early Stopping.
